# Santander Value Prediction — Baseline Models

> **Stage:** Benchmarking · **Status:** Completed initial holdout experiment

**Question:** How well can we predict transaction value without using *any* customer features?

**Why this matters:** Complex models should beat a simple, transparently evaluated reference. We compare the training-target mean, median, and **log-mean** using RMSLE.

**Protocol:** Random 80/20 train/validation split (`random_state=42`). Predictions are calculated from training targets only. These are **exploratory holdout results**, not a final untouched test.


## 1 · Load data and reproduce the split

Every notebook starts from its own imports and loading code so that **Run All** works from a fresh kernel. The dataset is fetched from [OpenML 42572](https://www.openml.org/search?id=42572&type=data).


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import train_test_split


In [2]:
dataset = fetch_openml(
    data_id=42572,
    target_column="target",
    as_frame=True,
)

X = dataset.data
y = dataset.target

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (4459, 4991)
y shape: (4459,)


In [3]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)


X_train: (3567, 4991)
X_val: (892, 4991)
y_train: (3567,)
y_val: (892,)


## 2 · Mean and median baselines

**Hypothesis:** The median may outperform the mean on a right-skewed target under RMSLE, though neither is guaranteed to minimize RMSLE.

- **Mean:** optimal constant prediction for minimizing *training MSE* in original units.
- **Median:** optimal constant prediction for minimizing *training MAE* in original units.

Both ignore `X` entirely. We create one prediction per validation observation with `np.full`.


In [4]:
mean_prediction = y_train.mean()
median_prediction = y_train.median()

print("Mean:", mean_prediction)
print("Median:", median_prediction)


Mean: 6031730.960521447
Median: 2256000.0


In [5]:
pred_mean = np.full(len(y_val), mean_prediction)
pred_median = np.full(len(y_val), median_prediction)

rmsle_mean = root_mean_squared_log_error(y_val, pred_mean)
rmsle_median = root_mean_squared_log_error(y_val, pred_median)

print(f"Mean Baseline RMSLE: {rmsle_mean:.4f}")
print(f"Median Baseline RMSLE: {rmsle_median:.4f}")


Mean Baseline RMSLE: 2.0351
Median Baseline RMSLE: 1.7021


**Finding:** Mean RMSLE = **2.0351**, median RMSLE = **1.7021** on this particular holdout. The median wins; this does not mean median is universally better for regression.


## 3 · Log-mean baseline

**Why:** RMSLE squares differences between `ln(1 + actual)` and `ln(1 + prediction)`; hence the constant that minimizes squared training error **in log space** is the mean of transformed training targets.

1. Compute `np.log1p(y_train)`.
2. Take the mean in log space.
3. Convert back with `np.expm1(...)`.
4. Score the resulting constant in the original target scale using RMSLE.

The conversion back produces the transaction-value prediction users expect, not a logarithm.


In [6]:
log_mean_prediction = np.expm1(np.log1p(y_train).mean())
pred_log_mean = np.full(len(y_val), log_mean_prediction)

rmsle_log_mean = root_mean_squared_log_error(y_val, pred_log_mean)

print(f"Log-Mean prediction: {log_mean_prediction:.2f}")
print(f"Log-Mean Baseline RMSLE: {rmsle_log_mean:.4f}")


Log-Mean prediction: 1964498.59
Log-Mean Baseline RMSLE: 1.6963


## 4 · Comparison and decision

The table is computed from the objects above, not manually retyped, so it updates when the experiment is rerun.


In [7]:
baseline_results = pd.DataFrame({
    "Model": ["Mean", "Median", "Log-Mean"],
    "Validation RMSLE": [rmsle_mean, rmsle_median, rmsle_log_mean],
})

display(baseline_results)


,Model,Validation RMSLE
0,Mean,2.035067
1,Median,1.702119
2,Log-Mean,1.696279


**Recorded holdout results (from the original run):**

| Constant strategy | Validation RMSLE |
|---|---:|
| Mean | 2.0351 |
| Median | 1.7021 |
| **Log-Mean** | **1.6963** |

**Interpretation:** Log-mean gives the smallest observed holdout RMSLE, slightly beating median. Its theoretical motivation is especially strong **for an RMSLE-targeted constant predictor**.

**Decision:** Carry the log-mean baseline into modeling; recompute it *within each CV training fold* for a fair comparison.

**Caution:** The validation set has already influenced our understanding of the project and must not be advertised as an untouched final test. Do not compare holdout scores with CV averages as though they used the same validation observations.
